<a href="https://colab.research.google.com/github/Park-wonil/BigdataPrac/blob/main/subway.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import folium
from folium.plugins import HeatMap

In [ ]:
station = pd.read_csv('/content/seoul-metro-station-info (1).csv')
data = pd.read_csv('/content/seoul-metro-2021.logs.csv')

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1941553 entries, 0 to 1941552
Data columns (total 4 columns):
 #   Column        Dtype 
---  ------        ----- 
 0   timestamp     object
 1   station_code  int64 
 2   people_in     int64 
 3   people_out    int64 
dtypes: int64(3), object(1)
memory usage: 59.3+ MB


In [ ]:
#승하차인원 그룹화
station_sum = data.groupby("station_code").sum()
station_sum

,timestamp,people_in,people_out
station_code,,,
150,2021-01-01T05:00:00.000+09:002021-01-01T06:00:...,12697273,12109991
151,2021-01-01T05:00:00.000+09:002021-01-01T06:00:...,5997344,6030491
152,2021-01-01T05:00:00.000+09:002021-01-01T06:00:...,9638952,9284693
153,2021-01-01T05:00:00.000+09:002021-01-01T06:00:...,7704599,7090896
154,2021-01-01T05:00:00.000+09:002021-01-01T06:00:...,6651283,6609055
...,...,...,...
2824,2021-01-01T05:00:00.000+09:002021-01-01T06:00:...,3206211,3097722
2825,2021-01-01T05:00:00.000+09:002021-01-01T06:00:...,1395919,1495603
2826,2021-01-01T05:00:00.000+09:002021-01-01T06:00:...,1614590,1475010


In [ ]:
station = station[['station.code','geo.latitude','geo.longitude']]

In [ ]:
station.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 285 entries, 0 to 284
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   station.code   285 non-null    int64  
 1   geo.latitude   285 non-null    float64
 2   geo.longitude  285 non-null    float64
dtypes: float64(2), int64(1)
memory usage: 6.8 KB


In [ ]:
station = station.set_index('station.code')

In [ ]:
station2 = station_sum.join(station)
station2

,timestamp,people_in,people_out,geo.latitude,geo.longitude
station_code,,,,,
150,2021-01-01T05:00:00.000+09:002021-01-01T06:00:...,12697273,12109991,37.554648,126.972559
151,2021-01-01T05:00:00.000+09:002021-01-01T06:00:...,5997344,6030491,37.564718,126.977108
152,2021-01-01T05:00:00.000+09:002021-01-01T06:00:...,9638952,9284693,37.570161,126.982923
153,2021-01-01T05:00:00.000+09:002021-01-01T06:00:...,7704599,7090896,37.571607,126.991806
154,2021-01-01T05:00:00.000+09:002021-01-01T06:00:...,6651283,6609055,37.570926,127.001849
...,...,...,...,...,...
2824,2021-01-01T05:00:00.000+09:002021-01-01T06:00:...,3206211,3097722,37.445210,127.156866
2825,2021-01-01T05:00:00.000+09:002021-01-01T06:00:...,1395919,1495603,37.440918,127.147564
2826,2021-01-01T05:00:00.000+09:002021-01-01T06:00:...,1614590,1475010,37.437428,127.140722


In [ ]:
#승차용 서울지도 그냥 folium으로하면 안나오네
seoul_in = folium.Map(
    location=[37.5517, 126.9917],
    tiles=None,
    zoom_start=12
)

folium.TileLayer(
    tiles="https://tile.openstreetmap.de/{z}/{x}/{y}.png",
    attr="© OpenStreetMap contributors",
    name="OpenStreetMap DE",
    max_zoom=19).add_to(seoul_in)
seoul_in

In [ ]:
HeatMap(data = station2[['geo.latitude','geo.longitude','people_in']]).add_to(seoul_in)
seoul_in

In [ ]:
seoul_out  = folium.Map(location=[37.55,126.98],zoom_start=12)
folium.TileLayer(
    tiles="https://tile.openstreetmap.de/{z}/{x}/{y}.png",
    attr="© OpenStreetMap contributors",
    name="OpenStreetMap DE",
    max_zoom=19).add_to(seoul_out)
HeatMap(data = station2[['geo.latitude','geo.longitude','people_out']]).add_to(seoul_out)
seoul_out

In [ ]:
#시간고려해서
morning = data[pd.to_datetime(data.timestamp).dt.hour<9]
morning

,timestamp,station_code,people_in,people_out
0,2021-01-01T05:00:00.000+09:00,150,86,85
1,2021-01-01T06:00:00.000+09:00,150,111,355
2,2021-01-01T07:00:00.000+09:00,150,157,438
3,2021-01-01T08:00:00.000+09:00,150,306,592
19,2021-01-01T05:00:00.000+09:00,151,43,40
...,...,...,...,...
1941518,2021-12-31T08:00:00.000+09:00,2827,391,311
1941534,2021-12-31T05:00:00.000+09:00,2828,20,10
1941535,2021-12-31T06:00:00.000+09:00,2828,83,41
1941536,2021-12-31T07:00:00.000+09:00,2828,279,119


In [ ]:
evening = data[pd.to_datetime(data.timestamp).dt.hour>17]
evening

,timestamp,station_code,people_in,people_out
13,2021-01-01T18:00:00.000+09:00,150,658,437
14,2021-01-01T19:00:00.000+09:00,150,579,425
15,2021-01-01T20:00:00.000+09:00,150,479,354
16,2021-01-01T21:00:00.000+09:00,150,510,307
17,2021-01-01T22:00:00.000+09:00,150,445,134
...,...,...,...,...
1941548,2021-12-31T19:00:00.000+09:00,2828,74,263
1941549,2021-12-31T20:00:00.000+09:00,2828,73,145
1941550,2021-12-31T21:00:00.000+09:00,2828,95,209
1941551,2021-12-31T22:00:00.000+09:00,2828,54,138


In [ ]:
print(morning.columns)

Index(['timestamp', 'station_code', 'people_in', 'people_out'], dtype='object')


In [ ]:
#그룹화한거랑 합치기
morning_sum =morning.groupby("station_code").sum()
evening_sum = evening.groupby("station_code").sum()

In [ ]:
#역 정보 합치기
morning_joined = pd.concat([morning_sum, station], axis=1)
evening_joined = pd.concat([evening_sum, station], axis=1)

In [ ]:
#출근시간
morning_seoul_in = folium.Map(
    location=[37.5517, 126.9917],
    tiles=None,
    zoom_start=12
)
folium.TileLayer(
    tiles="https://tile.openstreetmap.de/{z}/{x}/{y}.png",
    attr="© OpenStreetMap contributors",
    name="OpenStreetMap DE",
    max_zoom=19).add_to(morning_seoul_in)
heat_data = morning_joined[
    ['geo.latitude', 'geo.longitude', 'people_in']
].dropna().values.tolist()

HeatMap(data=heat_data,radius=25).add_to(morning_seoul_in)

morning_seoul_in

In [ ]:
evening_seoul_in = folium.Map(
    location=[37.5517, 126.9917],
    tiles=None,
    zoom_start=12
)
folium.TileLayer(
    tiles="https://tile.openstreetmap.de/{z}/{x}/{y}.png",
    attr="© OpenStreetMap contributors",
    name="OpenStreetMap DE",
    max_zoom=19).add_to(evening_seoul_in)
Heat_data = morning_joined[
    ['geo.latitude', 'geo.longitude', 'people_in']
].dropna().values.tolist() #결측값없애기
HeatMap(data=Heat_data).add_to(evening_seoul_in)
evening_seoul_in

시간정해서하기

In [ ]:

station = pd.read_csv('/content/seoul-metro-station-info (1).csv')
data['hour'] = pd.to_datetime(data['timestamp']).dt.hour
data8 = data[data['hour'] == 8].reset_index(drop=True)

data8in = data8.groupby('station_code')['people_in'].mean()
data8out = data8.groupby('station_code')['people_out'].mean()
data8inout = pd.concat([data8in, data8out], axis=1)

station['station_code'] = station['station.code']
station = station.set_index('station_code')
station_re = station[['geo.latitude', 'geo.longitude']]

final_df = pd.concat([data8inout, station_re], axis=1)
final_df = final_df.dropna()

In [ ]:
heat_data = final_df[['geo.latitude', 'geo.longitude', 'people_in']].values.tolist()
map = folium.Map(location=[37.55,126.98],zoom_start=12)
folium.TileLayer(
    tiles="https://tile.openstreetmap.de/{z}/{x}/{y}.png",
    attr="© OpenStreetMap contributors",
    name="OpenStreetMap DE",
    max_zoom=19).add_to(map)
HeatMap(
    data=heat_data,
    radius=25,                # 반지름
    blur=15,                  # 흐림 정도 설정
    max_val=final_df['people_in'].max()# 가중치의 최댓값을 기준으로 색상 강도 조절
).add_to(map)
map


/tmp/ipykernel_747/4027961956.py:8: UserWarning: The `max_val` parameter is no longer necessary. The largest intensity is calculated automatically.
  HeatMap(
